In [2]:
%pip install brapi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.4/136.4 kB 3.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import random as rd
import requests
import os
from datetime import datetime
from pandas.tseries.offsets import BDay
from brapi import Brapi
from getpass import getpass

d1 = (datetime.today() - BDay(1)).normalize()
d2 = (datetime.today() - BDay(2)).normalize()

##O objetivo desse projeto é criar uma carteira ficticia de um fundo de ações para demonstrar de forma prática como funcionam as métricas de risco de mercado.

In [4]:
#Antes de começarmos, preciso de um dicionario com ações da bolsa de valores do Brasil para que possamos fazer as simulações de risco de mercado.

acoes_b3 = {'YDUQ3': 'YDUQS PART','WEGE3': 'WEG','VIVA3': 'VIVARA S.A.','VBBR3': 'VIBRA','VAMO3': 'VAMOS','VALE3': 'VALE','USIM5': 'USIMINAS','UGPA3': 'ULTRAPAR','TOTS3': 'TOTVS','TIMS3': 'TIM','VIVT3': 'TELEF BRASIL','TAEE11': 'TAESA','SUZB3': 'SUZANO S.A.','SMFT3': 'SMART FIT','SLCE3': 'SLC AGRICOLA','CSNA3': 'SID NACIONAL','SANB11': 'SANTANDER BR','SBSP3': 'SABESP','RAIL3': 'RUMO S.A.','RDOR3': 'REDE D OR','RADL3': 'RAIADROGASIL','PSSA3': 'PORTO SEGURO','PRIO3': 'PETRORIO','RECV3': 'PETRORECSA','PETR4': 'PETROBRAS','PETR3': 'PETROBRAS','NATU3': 'NATURA','MULT3': 'MULTIPLAN','MRVE3': 'MRV','MOTV3': 'MOTIVA SA','BEEF3': 'MINERVA','MBRF3': 'MARFRIG','POMO4': 'MARCOPOLO','MGLU3': 'MAGAZ LUIZA','LREN3': 'LOJAS RENNER','RENT4': 'LOCALIZA','RENT3': 'LOCALIZA','KLBN11': 'KLABIN S/A','ITUB4': 'ITAUUNIBANCO','ITSA4': 'ITAUSA','ISAE4': 'ISA ENERGIA','IRBR3': 'IRBBRASIL RE','IGTI11': 'IGUATEMI S.A','HYPE3': 'HYPERA','HAPV3': 'HAPVIDA','GOAU4': 'GERDAU MET','GGBR4': 'GERDAU','FLRY3': 'FLEURY','EQTL3': 'EQUATORIAL','EGIE3': 'ENGIE BRASIL','ENEV3': 'ENEVA','ENGI11': 'ENERGISA','EMBJ3': 'EMBRAER','DIRR3': 'DIRECIONAL','CYRE4': 'CYRELA REALT','CYRE3': 'CYRELA REALT','CURY3': 'CURY S/A','CMIN3': 'CSNMINERACAO','CPFE3': 'CPFL ENERGIA','CSAN3': 'COSAN','CPLE3': 'COPEL','CSMG3': 'COPASA','COGN3': 'COGNA ON','CMIG4': 'CEMIG','CEAB3': 'CEA MODAS','CXSE3': 'CAIXA SEGURI','BPAC11': 'BTGP BANCO','BRAV3': 'BRAVA','BRKM5': 'BRASKEM','BBAS3': 'BRASIL','BRAP4': 'BRADESPAR','BBDC4': 'BRADESCO','BBDC3': 'BRADESCO','BBSE3': 'BBSEGURIDADE','B3SA3': 'B3','AZZA3': 'AZZAS 2154','AXIA7': 'AXIA ENERGIA','AXIA6': 'AXIA ENERGIA','AXIA3': 'AXIA ENERGIA','AURE3': 'AUREN','ASAI3': 'ASSAI','ABEV3': 'AMBEV S/A','ALOS3': 'ALLOS'}

In [5]:
#ok. Agora, com as ações posso começar a randomizar a carteira e criar as características do fundo.
ativos_selecionados = rd.sample(list(acoes_b3.keys()), 15)

In [16]:
#Agora preciso de algumas metricas reais como vol e correlações dos ativos e pra isso vou usar uma api gratuita chamada Brapi.
#Como a licença gratuita da API só me permite visualizar dados de 3 meses utilizarei esse padrão, mas na realidade o ideal seria utilizar um histórico de, pelo menos, 1 ano.
import os
from getpass import getpass

os.environ["BRAPI_KEY"] = getpass("Digite sua API key: ")

api_key = os.getenv("BRAPI_KEY")

if not api_key:
    raise ValueError("API key não configurada")

client = Brapi(api_key=api_key)

dados = []

for item in ativos_selecionados:
    quote = client.quote.retrieve(
        tickers=item,
        range='3mo',
        interval='1d',
        fundamental=True
    )

    for ativo in quote.results:
        for linha in ativo.historical_data_price:
            dados.append({
                'Sigla': ativo.symbol,
                'Date': linha.date,
                'PU': linha.close
            })

vols = pd.DataFrame(dados)
vols['Date'] = pd.to_datetime(vols['Date'], unit='s').dt.normalize()
vols = vols.sort_values(['Sigla', 'Date'])
vols['retorno'] = vols.groupby('Sigla')['PU'].pct_change()
vols['vol_acumulada'] = (
    vols.groupby('Sigla')['retorno']
    .expanding()
    .std()
    .reset_index(level=0, drop=True)
)
vols['vol_acumulada_anual'] = vols['vol_acumulada'] * (252 ** 0.5)
vols

Digite sua API key: ··········


,Sigla,Date,PU,retorno,vol_acumulada,vol_acumulada_anual
63,AXIA7,2026-01-08,49.43,NaN,NaN,NaN
64,AXIA7,2026-01-09,49.30,-0.002630,NaN,NaN
65,AXIA7,2026-01-12,49.03,-0.005477,0.002013,0.031954
66,AXIA7,2026-01-13,47.84,-0.024271,0.011759,0.186669
67,AXIA7,2026-01-14,49.00,0.024247,0.019978,0.317146
...,...,...,...,...,...,...
814,RENT3,2026-04-02,47.67,-0.001675,0.026269,0.417005
815,RENT3,2026-04-03,47.67,0.000000,0.026043,0.413413
816,RENT3,2026-04-06,47.14,-0.011118,0.025875,0.410757
817,RENT3,2026-04-07,47.15,0.000212,0.025659,0.407330


## Agora que eu tenho o DataFrame com os preços, retornos e volatilidades, posso juntar as quantidades que simulei para obter as exposições por ativo. No caso de ações a simulação fica mais fácil já que o fator de risco das ações costumam ser elas mesmas.

In [17]:
carteira_fundo = pd.DataFrame()
carteira_fundo["Ativo"] = ativos_selecionados
rd_qtde = rd.sample(range(-900, 9000), len(ativos_selecionados))
carteira_fundo["Qtde"] = rd_qtde
carteira_fundo["Data"] = d1

carteira_fundo = carteira_fundo.merge(vols, left_on="Ativo", right_on="Sigla")
carteira_fundo = carteira_fundo[carteira_fundo["Date"].dt.date == d2.date()]
carteira_fundo["Exposicao"] = carteira_fundo["Qtde"]*carteira_fundo["PU"]
pl_fundo = carteira_fundo["Exposicao"].sum()
carteira_fundo["% PL"] = carteira_fundo["Exposicao"]/pl_fundo
carteira_fundo.drop(columns=["Sigla", "Date","PU","retorno","vol_acumulada"], inplace=True)
carteira_fundo.reset_index(drop=True, inplace=True)
carteira_fundo

,Ativo,Qtde,Data,vol_acumulada_anual,Exposicao,% PL
0,FLRY3,3241,2026-04-07,0.270013,51596.72,0.030761
1,AXIA7,8222,2026-04-07,0.381153,468571.78,0.279355
2,RECV3,3755,2026-04-07,0.356156,52156.95,0.031095
3,MULT3,5535,2026-04-07,0.304264,175846.95,0.104837
4,EMBJ3,1217,2026-04-07,0.456118,99392.39,0.059256
5,BEEF3,4613,2026-04-07,0.449988,19743.64,0.011771
6,GOAU4,4945,2026-04-07,0.315891,43268.75,0.025796
7,RADL3,3000,2026-04-07,0.326226,65580.00,0.039098
8,MGLU3,49,2026-04-07,0.623620,432.18,0.000258
9,HAPV3,2262,2026-04-07,0.587336,24158.16,0.014403


In [18]:
carteira_bench = pd.DataFrame()
carteira_bench["Ativo"] = ativos_selecionados
rd_qtde = rd.sample(range(-900, 9000), len(ativos_selecionados))
carteira_bench["Qtde"] = rd_qtde
carteira_bench["Data"] = d1

carteira_bench = carteira_bench.merge(vols, left_on="Ativo", right_on="Sigla")
carteira_bench = carteira_bench[carteira_bench["Date"].dt.date == d2.date()]
carteira_bench["Exposicao"] = carteira_bench["Qtde"]*carteira_bench["PU"]
pl_bench = carteira_bench["Exposicao"].sum()
carteira_bench["% PL"] = carteira_bench["Exposicao"]/pl_bench
carteira_bench.drop(columns=["Sigla", "vol_acumulada_anual", "Exposicao", "Date","PU","retorno","vol_acumulada"], inplace=True)
carteira_bench.reset_index(drop=True, inplace=True)
carteira_bench

,Ativo,Qtde,Data,% PL
0,FLRY3,-726,2026-04-07,-0.008905
1,AXIA7,792,2026-04-07,0.034778
2,RECV3,1034,2026-04-07,0.011066
3,MULT3,5437,2026-04-07,0.133093
4,EMBJ3,4314,2026-04-07,0.271470
5,BEEF3,11,2026-04-07,0.000036
6,GOAU4,6091,2026-04-07,0.041065
7,RADL3,3953,2026-04-07,0.066582
8,MGLU3,8133,2026-04-07,0.055271
9,HAPV3,3057,2026-04-07,0.025156


###  Agora que já evolui com as vols e exposições, posso fazer as primeiras medidas de risco. Então vamos focar em cenários e resultados diários de Stress!
###   O primeiro passo é o Stress Descorrelacionado, esse cenário é o mais facil pois posso pegar quantis relacionados ao melhor e pior rendimento dos ativos no período analisado.
###   No entanto, terei que dar um passo mais complexo para obter o cenário de Stress correlacionado. Como eu não tenho acesso a um longo histórico de preços, não poderei escolher cenários realistas como Joesley Day, Pandemia, entre outros.
###   Portanto, usarei um dia aleatório, já que o princípio é o mesmo e em um caso real eu apenas utilizaria um dos dias já citados.

In [19]:
def criar_cenarios_com_posicao(row):
    ret = row['retorno']
    qtd = row['Qtde']

    if qtd >= 0:
        # posição comprada → cenário pessimista é retorno negativo
        if ret < 0:
            return pd.Series({'cen_pess_res': ret, 'cen_otim_res': -ret})
        else:
            return pd.Series({'cen_pess_res': -ret, 'cen_otim_res': ret})
    else:
        # posição vendida → inverte o sinal dos cenários
        if ret > 0:
            return pd.Series({'cen_pess_res': ret, 'cen_otim_res': -ret})
        else:
            return pd.Series({'cen_pess_res': -ret, 'cen_otim_res': ret})

def ajustar_cenarios_desc(row):
    pess = row['cen_pess_descorrel']
    otim = row['cen_otim_descorrel']
    qtd = row['Qtde']

    if qtd >= 0:
        # posição comprada: mantém cenário
        return pd.Series({'cen_pess_desc': pess, 'cen_otim_desc': otim})
    else:
        # posição vendida: inverte sinais
        return pd.Series({'cen_pess_desc': otim, 'cen_otim_desc': pess})


#### Stress_Descorrel
stress_descorrel = (
    vols.groupby('Sigla')['retorno']
    .quantile([0.05, 0.95])
    .unstack()
    .rename(columns={'Sigla': 'Ativo', 0.05: 'cen_pess_descorrel', 0.95: 'cen_otim_descorrel'})
    .reset_index()
)
stress_descorrel = stress_descorrel.merge(
    carteira_fundo[['Ativo', 'Qtde']],
    left_on="Sigla", right_on='Ativo',
    how='inner'
)
cenarios_res = stress_descorrel.apply(ajustar_cenarios_desc, axis=1)
stress_descorrel = pd.concat([stress_descorrel.drop(columns=['cen_pess_descorrel','cen_otim_descorrel','Ativo','Qtde']), cenarios_res], axis=1)
stress_descorrel["Cenario_desc_Hoje"] = np.where(stress_descorrel["cen_pess_desc"] < stress_descorrel["cen_otim_desc"],"Pessimista", "Otimista")

#### Stress_Correl
data_aleatoria = np.random.choice(vols['Date'].unique())
stress_correl = vols.loc[vols['Date'] == data_aleatoria, ['Sigla', 'retorno']]
def criar_cenarios(ret):
    if ret < 0:
        return pd.Series({'cen_pess': ret, 'cen_otim': -ret})
    else:
        return pd.Series({'cen_pess': -ret, 'cen_otim': ret})

cenarios = stress_correl['retorno'].apply(criar_cenarios)
stress_correl = pd.concat([stress_correl, cenarios], axis=1)

stress_correl = stress_correl.merge(
    carteira_fundo[['Ativo', 'Qtde']],
    left_on="Sigla", right_on='Ativo',
    how='inner'
)

cenarios = stress_correl.apply(criar_cenarios_com_posicao, axis=1)
stress_correl = pd.concat([stress_correl.drop(columns=['retorno','cen_otim','cen_pess','Ativo','Qtde']), cenarios], axis=1)
stress_correl = stress_correl.rename(columns={'cen_pess_res': 'cen_pess_correl','cen_otim_res': 'cen_otim_correl'})
stress_correl["Cenario_correl_Hoje"] = np.where(stress_correl["cen_pess_correl"] < stress_correl["cen_otim_correl"],"Pessimista", "Otimista")


stress = stress_descorrel.merge(stress_correl, on="Sigla")

stress_consolidado = stress.merge(carteira_fundo, left_on="Sigla", right_on="Ativo")

stress_consolidado['cenario_valor_desc'] = stress_consolidado.apply(
    lambda row: row['cen_pess_desc'] if row['Cenario_desc_Hoje'] == 'Pessimista'
    else row['cen_otim_desc'],
    axis=1
)

stress_consolidado['cenario_valor_correl'] = stress_consolidado.apply(
    lambda row: row['cen_pess_correl'] if row['Cenario_correl_Hoje'] == 'Pessimista'
    else row['cen_otim_correl'],
    axis=1
)


stress_consolidado['Stress_Descorrel'] = (
    stress_consolidado['% PL'] * stress_consolidado['cenario_valor_desc']
)
stress_consolidado['Stress_Correl'] = (
    stress_consolidado['% PL'] * stress_consolidado['cenario_valor_correl']
)
stress_consolidado.drop(columns=["Sigla", "Qtde","Exposicao","cenario_valor_desc","cenario_valor_correl","vol_acumulada_anual"], inplace=True)
stress_consolidado

,cen_pess_desc,cen_otim_desc,Cenario_desc_Hoje,cen_pess_correl,cen_otim_correl,Cenario_correl_Hoje,Ativo,Data,% PL,Stress_Descorrel,Stress_Correl
0,-0.040654,0.044572,Pessimista,-0.003888,0.003888,Pessimista,AXIA7,2026-04-07,0.279355,-0.011357,-0.001086
1,-0.048965,0.041548,Pessimista,-0.035948,0.035948,Pessimista,BEEF3,2026-04-07,0.011771,-0.000576,-0.000423
2,-0.029535,0.035450,Pessimista,-0.019679,0.019679,Pessimista,CPLE3,2026-04-07,0.053147,-0.001570,-0.001046
3,-0.021687,0.021869,Pessimista,-0.007654,0.007654,Pessimista,CXSE3,2026-04-07,0.062084,-0.001346,-0.000475
4,-0.045718,0.044015,Pessimista,-0.000851,0.000851,Pessimista,EMBJ3,2026-04-07,0.059256,-0.002709,-0.000050
5,-0.035192,0.039524,Pessimista,-0.096627,0.096627,Pessimista,ENEV3,2026-04-07,0.044411,-0.001563,-0.004291
6,-0.028016,0.024237,Pessimista,-0.002361,0.002361,Pessimista,FLRY3,2026-04-07,0.030761,-0.000862,-0.000073
7,-0.029645,0.033247,Pessimista,-0.004016,0.004016,Pessimista,GOAU4,2026-04-07,0.025796,-0.000765,-0.000104
8,-0.050331,0.053501,Pessimista,-0.010480,0.010480,Pessimista,HAPV3,2026-04-07,0.014403,-0.000725,-0.000151
9,-0.062551,0.058794,Pessimista,-0.033728,0.033728,Pessimista,MGLU3,2026-04-07,0.000258,-0.000016,-0.000009


Meu último passo é criar o cálculo de VaR. Por questões de padronização eu preferi utilizar o VaR 95% e pra isso preciso, também, criar minha própria matriz de covariancia de fatores! Isso vai deixar tudo mais próximo de um motor de calculo real.

In [20]:
ret_pivot = vols.pivot(index="Date", columns="Sigla", values="retorno").dropna()
cov_matrix = ret_pivot.cov() * 252

weights = carteira_fundo.set_index("Ativo")["% PL"]
weights = weights.reindex(cov_matrix.columns).fillna(0)
weights = weights / weights.sum()

vol_carteira = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
z = 1.65  # 95%
var_fundo = z * vol_carteira
print(var_fundo)

0.4247629786054201


In [21]:
fundo = pd.DataFrame(index=[0])
fundo["Nome"] = "Aoba"
fundo["Data"] = d1
fundo["BVaR"] = var_fundo
fundo["Stress Correlacionado"] = round(np.abs(stress_consolidado["Stress_Correl"].sum()),4)
fundo["Stress Descorrelacionado"] = round(np.abs(stress_consolidado["Stress_Descorrel"].sum()),4)
fundo["CNPJ"] = "00.000.000/0000-00"
fundo["Classe"] = "FIA"
fundo["Condominio"] = "Aberto"
fundo["Benchmark"] = "IAOBA"
fundo["PL"] = round(pl_fundo,4)
fundo["Qtde cotas"] = rd.randint(9000000,90000000)
fundo["Cota"] = round(fundo["Qtde cotas"]/fundo["PL"],4)
fundo

,Nome,Data,BVaR,Stress Correlacionado,Stress Descorrelacionado,CNPJ,Classe,Condominio,Benchmark,PL,Qtde cotas,Cota
0,Aoba,2026-04-07,0.424763,0.0084,0.0328,00.000.000/0000-00,FIA,Aberto,IAOBA,1677336.04,36736610,21.9018
